## What is REINVENT4?

**REINVENT4** REINVENT is a molecular design tool for de novo design, scaffold hopping, R-group replacement, linker design, molecule optimization, and other small molecule design tasks. REINVENT uses a Reinforcement Learning (RL) algorithm to generate optimized molecules compliant with a user defined property profile defined as a multi-component score. Transfer Learning (TL) can be used to create or pre-train a model that generates molecules closer to a set of input molecules.

Key concepts you’ll see in this tutorial:

- **Prior**: a pre-trained generative model (baseline chemistry knowledge).
- **Agent**: a copy of the prior that is *updated during RL* to generate better-scoring molecules.
- **Scoring function**: a set of components (property filters/objectives) combined into a single score, typically in 
  a weighted geometric mean.
- **Staged learning**: a run can be broken into stages with different scoring objectives or settings.
- **Outputs**: REINVENT writes logs, TensorBoard traces, checkpoints, and CSV summaries of generated molecules.

## Goals


- Run a small **staged learning** RL job using simple scoring components (alerts + a property objective).
- Inspect the generated molecules (CSV) with `pandas` and visualize them with `mols2grid`.
- Run a second example with a diversity filter and a different reward.
- Demonstrate **sampling** from a trained checkpoint and **scoring** an external SMILES file.

**For moreindormation:** Visit [the GitHub](https://github.com/MolecularAI/REINVENT4), or [the paper](https://doi.org/10.1186/s13321-024-00812-5)

In [ ]:
import workshop_setup


In [ ]:
# Checking the installation
!reinvent --help

In [ ]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors
import mols2grid
import os


### Build a REINVENT4 config (TOML)

REINVENT4 is driven by a **TOML configuration**. In the next cell we’ll assemble one as a Python string and write it to `configs/reinvent_rl_config.toml`.

What to pay attention to:

- **`run_type = "staged_learning"`**: runs RL.
- **`[parameters]`**:
  - `prior_file`: the starting model
  - `agent_file`: model that will be updated (often initialized from the prior)
  - `summary_csv_prefix`: where REINVENT writes per-step molecule summaries
  - `batch_size`: molecules sampled per RL step
- **`[learning_strategy]`**: RL algorithm/settings (here: DAP + `sigma` controls reward shaping strength).
- **`[[stage]]`**: one RL stage with stop criteria (`min_steps`, `max_steps`).
- **`[stage.scoring]` and `[[stage.scoring.component]]`**: defines the scoring function.

This first example uses:

- **`custom_alerts`**: penalize/avoid undesired SMARTS patterns.
- **`MolecularWeight`**: push molecules into a target MW window using a transform (double-sigmoid).

In [ ]:
global_parameters = """
run_type = "staged_learning"
device = "cpu"
tb_logdir = "tb_logs_sigma128_unusualmw"
json_out_config = "_stage1.json"
"""

prior_filename = os.path.join("priors/reinvent.prior")
agent_filename = prior_filename
output_folder = os.path.join(os.getcwd(), "data", "output")
parameters = f"""
[parameters]

prior_file = "{prior_filename}"
agent_file = "{agent_filename}"
summary_csv_prefix = "reinvent_rl"  # You can choose your own prefix for the output csv files, e.g. "stage_numrotbond"  instead of reinvent_rl

batch_size = 64

use_checkpoint = false
"""


learning_strategy = """
[learning_strategy]

type = "dap"
sigma = 128
rate = 0.0001
"""

stages = """
[[stage]]

max_score = 1.0
min_steps = 5
max_steps = 10

chkpt_file = 'stage1.chkpt'

[stage.scoring]
type = "geometric_mean"

[[stage.scoring.component]]
[stage.scoring.component.custom_alerts]

[[stage.scoring.component.custom_alerts.endpoint]]
name = "Alerts"

params.smarts = [
    "[*;r8]",
    "[*;r9]",
    "[*;r10]",
    "[*;r11]",
    "[*;r12]",
    "[*;r13]",
    "[*;r14]",
    "[*;r15]",
    "[*;r16]",
    "[*;r17]",
    "[#8][#8]",
    "[#6;+]",
    "[#16][#16]",
    "[#7;!n][S;!$(S(=O)=O)]",
    "[#7;!n][#7;!n]",
    "C#C",
    "C(=[O,S])[O,S]",
    "[#7;!n][C;!$(C(=[O,N])[N,O])][#16;!s]",
    "[#7;!n][C;!$(C(=[O,N])[N,O])][#7;!n]",
    "[#7;!n][C;!$(C(=[O,N])[N,O])][#8;!o]",
    "[#8;!o][C;!$(C(=[O,N])[N,O])][#16;!s]",
    "[#8;!o][C;!$(C(=[O,N])[N,O])][#8;!o]",
    "[#16;!s][C;!$(C(=[O,N])[N,O])][#16;!s]"
]
[[stage.scoring.component]]
[stage.scoring.component.MolecularWeight]

[[stage.scoring.component.MolecularWeight.endpoint]]
name = "Molecular weight"  # user chosen name for output
weight = 1  # weight to fine-tune the relevance of this component

transform.type = "double_sigmoid"
transform.high = 1200.0
transform.low = 700.0
transform.coef_div = 1200.0
transform.coef_si = 20.0
transform.coef_se = 20.0
"""



In [ ]:
config = global_parameters + parameters + learning_strategy + stages

toml_config_filename = "configs/reinvent_rl_config.toml"

with open(toml_config_filename, "w") as tf:
    tf.write(config)

### Run REINVENT4 

The next command runs REINVENT4 using the TOML config you just wrote.

What you should see created/updated:

- **Log file** (e.g., `stage1.log`) with progress messages.
- **TensorBoard directory** (`tb_logdir...`) with learning curves and component traces.
- **Checkpoint** (`stage1.chkpt`) containing the trained agent state.
- **CSV summaries** prefixed by `summary_csv_prefix` (generated SMILES + scores/components).

In [ ]:
!python -m reinvent.Reinvent -l stage1.log $toml_config_filename

### Inspect learning with TensorBoard

TensorBoard helps you understand whether RL is doing something sensible.

Typical signals to look at:

- **Overall score trend**: does it increase over steps?
- **Component values**: are you improving the intended objective (e.g., MW in the target region) without collapsing diversity?
- **Warnings**: repeated failures can indicate invalid SMILES, overly strict alerts, or an impossible objective.

In [ ]:
!tensorboard  --logdir tb_logs_sigma128_unusualmw_0 --load_fast true

### Second example: reward + diversity filter

This second configuration demonstrates two common additions:

- **A different scoring component** (`NumRotBond`) with a **reverse-sigmoid** transform (reward fewer rotatable bonds).
- A **diversity filter** to discourage generating the same scaffold repeatedly.

The goal is to show that in REINVENT you often need *both*:

- **an objective** (what you want)
- **a diversity mechanism** (so you don’t get 100 near-duplicates)

We’ll write a new TOML file and run RL again, producing a new checkpoint and TensorBoard traces.

In [ ]:
global_parameters = """
run_type = "staged_learning"
device = "cpu"
tb_logdir = "tb_logs_numrotbond"
json_out_config = "_stage1.json"
"""

prior_filename = os.path.join("./priors/reinvent.prior")
agent_filename = prior_filename
output_folder = os.path.join(os.getcwd(), "data", "output")
parameters = f"""
[parameters]

prior_file = "{prior_filename}"
agent_file = "{agent_filename}"
summary_csv_prefix = "stage_numrotbond"  # You can choose your own prefix for the output csv files, e.g. "stage_numrotbond"

batch_size = 100

use_checkpoint = false
"""


learning_strategy = """
[learning_strategy]

type = "dap"
sigma = 128
rate = 0.0001

[diversity_filter]

type = "IdenticalMurckoScaffold" # IdenticalTopologicalScaffold,
                                 # ScaffoldSimilarity, PenalizeSameSmiles
bucket_size = 10                 # memory size in number of compounds
minscore = 0.4                   # only memorize if this threshold is exceeded
minsimilarity = 0.4              # minimum similarity for ScaffoldSimilarity
penalty_multiplier = 0.5         # penalty factor for PenalizeSameSmiles


"""

stage = """
[[stage]]

max_score = 1.0
min_steps = 5
max_steps = 10

chkpt_file = 'numrotbond_reversesig.chkpt'

[stage.scoring]
type = "geometric_mean"

######keep the alerts to avoid unwanted structures

[[stage.scoring.component]]
[stage.scoring.component.custom_alerts]

[[stage.scoring.component.custom_alerts.endpoint]]
name = "Alerts"

params.smarts = [
    "[*;r8]",
    "[*;r9]",
    "[*;r10]",
    "[*;r11]",
    "[*;r12]",
    "[*;r13]",
    "[*;r14]",
    "[*;r15]",
    "[*;r16]",
    "[*;r17]",
    "[#8][#8]",
    "[#6;+]",
    "[#16][#16]",
    "[#7;!n][S;!$(S(=O)=O)]",
    "[#7;!n][#7;!n]",
    "C#C",
    "C(=[O,S])[O,S]",
    "[#7;!n][C;!$(C(=[O,N])[N,O])][#16;!s]",
    "[#7;!n][C;!$(C(=[O,N])[N,O])][#7;!n]",
    "[#7;!n][C;!$(C(=[O,N])[N,O])][#8;!o]",
    "[#8;!o][C;!$(C(=[O,N])[N,O])][#16;!s]",
    "[#8;!o][C;!$(C(=[O,N])[N,O])][#8;!o]",
    "[#16;!s][C;!$(C(=[O,N])[N,O])][#16;!s]"
]


[[stage.scoring.component]]
[stage.scoring.component.NumRotBond]

[[stage.scoring.component.NumRotBond.endpoint]]
name = "ROTB"
weight = 1
transform.type = "reverse_sigmoid"
transform.high = 20
transform.low = 0
transform.k = 0.5

"""

In [ ]:
config = global_parameters + parameters+ learning_strategy + stage
toml_config_filename = "configs/reward_numrotb.toml"
with open(toml_config_filename, 'w') as tf:
    tf.write(config)


In [ ]:
!python -m reinvent.Reinvent -l numrotb.log $toml_config_filename

In [ ]:
!tensorboard --logdir tb_logs_numrotbond_0 --load_fast true

### Sampling from a trained checkpoint

After RL, you typically want to **sample** a larger set of molecules from the trained agent (checkpoint).

In REINVENT4, `run_type = "sampling"`:

- loads `model_file` (a `.chkpt` you trained)
- generates `num_smiles`
- writes them to `output_file` as SMILES

This is the main bridge to downstream workflows: the sampled molecules become the input library for docking, filtering, or further analysis.

In [ ]:
global_sampling_parameters= """
run_type = "sampling"
device = "cpu"
json_out_config = "_sampling.json"
"""
parameters = """
[parameters]
model_file = "numrotbond_reversesig.chkpt" ##
output_file = 'sampling_numrotbond.csv'
num_smiles = 1200
unique_molecules = true
randomize_smiles = true
"""

In [ ]:
samp_conf = global_sampling_parameters + parameters
samp_conf_filename = "configs/sampling_numrotbond.toml"
with open(samp_conf_filename, "w") as tf:
    tf.write(samp_conf)

In [ ]:

!python -m reinvent.Reinvent -l samplin.log $samp_conf_filename

### Scoring an external SMILES file

REINVENT can also be used as a **scoring engine** without RL.

With `run_type = "scoring"` you provide:

- `smiles_file`: a CSV of SMILES (e.g., from sampling)
- `output_csv`: a scored CSV containing the total score and component values

This is useful when you want to:

- compare different scoring functions on the same molecule set
- score molecules generated by another method
- do post-hoc analysis without retraining

In [ ]:

global_scoring_parameters = """
run_type = "scoring"
json_out_config = "_scoring.json"
"""

parameters = """
[parameters]
smiles_file = "sampling_numrotbond.csv"   #the path of sampled molecules
output_csv = "scoring_numrotbond.csv"
"""
scoring = """
[scoring]
type = "geometric_mean"

[[scoring.component]]
[scoring.component.GraphLength]
[[scoring.component.GraphLength.endpoint]]
name = "GraphLenght" #number of bonds in longest path
weight = 0.2
transform.type = "reverse_sigmoid"
transform.high = 40
transform.low = 20
transform.k = 0.5

######
#  Add scoring functions as many as you want. Define the weights for each score function and tranform them properly.
######



"""


In [ ]:
scoring_conf = global_scoring_parameters + parameters + scoring
with open("scoring_numrotbond.toml", "w") as tf:
    tf.write(scoring_conf)

In [ ]:
!python -m reinvent.Reinvent -l scoring.log scoring_numrotbond.toml

In [ ]:
scored_mols = pd.read_csv("scoring_numrotbond.csv")

# Convert SMILES to RDKit molecule objects for visualization
scored_mols['mols'] = [Chem.MolFromSmiles(smi) for smi in scored_mols["SMILES"]]

# Sort by score (best molecules first)
scored_mols = scored_mols.sort_values('Score', ascending=False).reset_index(drop=True)

# Interactive molecular grid
mols2grid.display(
    scored_mols,
    mol_col='mols',
    dpi=300,
    subset=['Score'],  # You can also add the scoring components, if you want to analyse them too.
    transform={
        "Score": lambda x: f"{x:.2f}" # add the other scoring components to print transformed values instead of long, raw digits
    },
    n_items_per_page=12,
    size=(200, 200),
)
